# PathSnap: 02. Candidate Matching Model Training & Hyperparameter Tuning
### Multi-Model Benchmarking Across 6 Algorithm Families & Bayesian Optimization

This notebook executes the complete candidate matching and machine learning workflow:
1. **Road Arc Candidate Generation**: KDTree spatial indexing ($O(\log N)$ spatial query).
2. **Feature Engineering**: 8 kinematic, geometric, and topological features.
3. **Benchmarking 6 Algorithm Families**:
   - Linear / Logistic: `LogisticRegression`
   - Instance-based: `KNeighborsClassifier`
   - Tree-based: `DecisionTreeClassifier`
   - Ensemble: `RandomForestClassifier`
   - Boosting: `GradientBoostingClassifier`
   - Neural Network: `MLPClassifier` (Multi-Layer Perceptron)
4. **Bayesian Hyperparameter Optimization**: Optuna TPESampler optimizing blended F1 + Accuracy.
5. **Artifact Persistence**: Saving production model and preprocessing pipelines.


In [ ]:
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path(".").resolve().parent))

from src.data.loader import TripLoader
from src.features.candidates import CandidateGenerator
from src.features.engineering import FeatureExtractor
from src.models.train import get_model_pipeline, save_model
from src.models.evaluate import evaluate_predictions
from src.models.tune import run_bayesian_tuning

loader = TripLoader("../data")
cg = CandidateGenerator(search_radius_m=120.0, top_k=6)
fe = FeatureExtractor()


## 1. Feature Extraction from Trajectories
For each GPS point, we generate spatial candidate arcs within 120m and compute 8 discriminative features:
1. `distance_to_arc` (metric orthogonal distance)
2. `heading_difference` (angular alignment $|\theta_{\text{GPS}} - \theta_{\text{arc}}|$)
3. `speed` (instantaneous GPS speed)
4. `length_of_arc` (physical road segment length)
5. `delta_distance` (displacement difference)
6. `arc_connectivity_prior` (topological degree)
7. `normalized_distance` (distance / search radius)
8. `temporal_dt` (sampling interval)


In [ ]:
train_trips = loader.list_trips("train")[:15]
test_trips = loader.list_trips("student_test")[:6]

train_dfs = []
for tid in train_trips:
    t = loader.load_trip(tid, "train")
    trk = t["track"][:250]
    cands = cg.generate_candidates(trk, t["arcs"], t["nodes"])
    df = fe.extract_features_for_trip(trk, t["arcs"], t["nodes"], cands, route=t["route"])
    df["trip_id"] = tid
    train_dfs.append(df)
df_train = pd.concat(train_dfs, ignore_index=True)

test_dfs = []
for tid in test_trips:
    t = loader.load_trip(tid, "student_test")
    trk = t["track"][:200]
    cands = cg.generate_candidates(trk, t["arcs"], t["nodes"])
    df = fe.extract_features_for_trip(trk, t["arcs"], t["nodes"], cands, route=t["route"])
    df["trip_id"] = tid
    test_dfs.append(df)
df_test = pd.concat(test_dfs, ignore_index=True)

print(f"Extracted {len(df_train)} training candidate pairs and {len(df_test)} test candidate pairs.")
print(f"Match ratio: {df_train['is_match'].mean():.2%}")


## 2. Benchmark Model Comparison Across 6 Families


In [ ]:
comp_df = pd.read_csv("../results/model_comparison.csv")
comp_df[["Model", "Algorithm Family", "CV Macro F1-score", "Top-1 Accuracy", "Route Continuity", "Log Loss", "Inference Time (ms/obs)"]]


## 3. Visualizing Model Performance


In [ ]:
plt.figure(figsize=(10, 4.5))
colors = ["#3B82F6", "#10B981", "#F59E0B", "#EF4444", "#8B5CF6", "#EC4899"]
bars = plt.bar(comp_df["Model"], comp_df["Top-1 Accuracy"], color=colors[:len(comp_df)])
plt.ylabel("Observation Top-1 Accuracy")
plt.title("Candidate Road Selection: Observation Top-1 Accuracy across Models")
plt.ylim(0, 1.0)
plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 0.02, f"{h*100:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Bayesian Hyperparameter Optimization Results (Optuna)
Iterative optimization records maximizing the blended objective: $0.6 \times \text{Macro F1} + 0.4 \times \text{Accuracy}$.


In [ ]:
tuning_df = pd.read_csv("../results/tuning_results.csv")
print(f"Loaded {len(tuning_df)} Bayesian optimization trials.")
tuning_df.sort_values(by="blended_objective", ascending=False).head(8)


In [ ]:
# Plot Optimization Convergence
plt.figure(figsize=(8, 4))
plt.plot(tuning_df["trial"], tuning_df["blended_objective"], marker="o", linestyle="-", color="#3B82F6", alpha=0.6, label="Trial Score")
plt.plot(tuning_df["trial"], tuning_df["blended_objective"].cummax(), marker="^", linestyle="--", color="#10B981", linewidth=2, label="Current Best")
plt.title("PathSnap: Bayesian Hyperparameter Convergence")
plt.xlabel("Trial / Iteration")
plt.ylabel("Blended Objective")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()
